# 11. Offensive-zone recovery pace and shot generation


### Purpose

Check whether 5v5 possessions that start with an offensive-zone puck recovery generate a shot, and how that outcome relates to puck-movement pace after the recovery. The coaching question is: **can a team turn offensive-zone recoveries into shots while moving the puck quickly?**

### Expected outcome

One record per qualifying recovery-started possession, pooled slower/middle/faster pace bands, shot-generation bar charts, and team and sequence examples. Keep unknown pace in coverage and distinguish retrospective association from prediction.


## 11.1. Setup and load processed data

Use the same augmented events, transition pace, and reference tables as notebooks 07-10. Include 5v5 only and pool sources. Inferred possession boundaries and modeled timing retain their existing limitations.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 11.2. Select possessions starting with an offensive-zone recovery

Require the first event of the inferred possession sequence to be `Puck Recovery`, credited to the sequence owner, at 5v5, in the offensive zone (`x > 125`). Do not include recoveries occurring partway through a sequence or infer a new sequence around every recovery.

Success is at least one `Shot` or `Goal` by the recovering team later in that same sequence. Include blocked and missed attempts and exclude penalty-shot goals. This outcome lasts until the sequence ends, rather than using notebook 07's five-second entry window. Sequence endings follow ownership, manpower, restart, and period rules; they do not perfectly identify every stoppage.


In [2]:
sequence_starts = events.groupby("possession_id", sort=False).head(1)
recoveries = sequence_starts.loc[
    sequence_starts["event"].eq("Puck Recovery")
    & sequence_starts["is_5v5"]
    & sequence_starts["event_team_zone"].eq("OZ")
    & sequence_starts["team_id"].eq(sequence_starts["possession_team_id"])
].copy()
recoveries["team_name"] = recoveries["team_id"].map(teams.set_index("team_id")["team_name"])
recoveries["player_name"] = recoveries["player_id"].map(players.set_index("player_id")["player_name"])
recoveries["game_date"] = recoveries["game_id"].map(games.set_index("game_id")["game_date"])
sequence_events = {pid: group for pid, group in events.loc[
    events["possession_id"].isin(recoveries["possession_id"])
].groupby("possession_id", sort=False)}
assert all(group["is_5v5"].all() for group in sequence_events.values())
print("Qualifying offensive-zone recovery possessions:", len(recoveries))


Qualifying offensive-zone recovery possessions: 1538


## 11.3. Calculate pace after recovery, before the first shot

For shot-generating possessions, pool eligible movements after recovery with end event IDs strictly before the first shot. Exclude movement ending at the shot, the shot itself, and all later play, matching the pre-event convention in the other notebooks. For possessions without a shot, pool all eligible movement after recovery through the sequence end.

Keep one outcome per possession. Record movement count/duration, observed sequence span, first-shot timing, passes before the cutoff, and first-shot distance. Immediate shots can have no measurable pre-shot pace and remain in coverage rather than being assigned zero.

**This is a retrospective comparison.** The first shot determines when measurement stops for successes, while unsuccessful possessions end at a different boundary. The observation windows differ and longer sequences have more opportunity to generate shots. The chart cannot establish that faster movement increases shot probability. A future predictive analysis would need a fixed initial observation window followed by a separate outcome window.


In [3]:
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()]
movement_groups = {
    pid: group[["start_event_id", "end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
recovery_rows = []
for recovery in recoveries.itertuples(index=False):
    sequence = sequence_events[recovery.possession_id]
    shots = sequence.loc[
        sequence["event_id"].gt(recovery.event_id)
        & sequence["team_id"].eq(recovery.team_id)
        & sequence["event"].isin(["Shot", "Goal"])
    ]
    has_shot = not shots.empty
    first_shot = shots.iloc[0] if has_shot else None
    cutoff_id = int(first_shot["event_id"]) if has_shot else int(sequence["event_id"].max()) + 1
    movement = movement_groups.get(recovery.possession_id)
    if movement is not None:
        movement = movement[(movement[:, 0] >= recovery.event_id) & (movement[:, 1] < cutoff_id)]
    count = 0 if movement is None else len(movement)
    distance = 0.0 if movement is None else float(movement[:, 2].sum())
    duration = 0.0 if movement is None else float(movement[:, 3].sum())
    shot_distance = np.nan
    if has_shot and pd.notna(first_shot["x"]) and pd.notna(first_shot["y"]):
        if 0 <= first_shot["x"] <= 200 and 0 <= first_shot["y"] <= 85:
            shot_distance = float(np.hypot(first_shot["x"] - 189, first_shot["y"] - 42.5))
    recovery_rows.append({
        "event_id": recovery.event_id, "leads_to_shot": has_shot,
        "shot_attempts": len(shots),
        "first_shot_event_id": int(first_shot["event_id"]) if has_shot else None,
        "time_to_first_shot_seconds": float(recovery.modeled_clock_seconds - first_shot["modeled_clock_seconds"]) if has_shot else np.nan,
        "first_shot_distance_ft": shot_distance,
        "sequence_span_seconds": float(recovery.modeled_clock_seconds - sequence["modeled_clock_seconds"].min()),
        "movement_transitions": count, "movement_distance_ft": distance,
        "movement_seconds": duration,
        "post_recovery_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "pace_status": "eligible" if duration > 0 else "no_movement" if count == 0 else "zero_duration",
        "passes_before_cutoff": int((sequence["event"].isin(["Pass", "Incomplete Pass"])
                                    & sequence["event_id"].lt(cutoff_id)).sum()),
    })
recoveries = recoveries.merge(pd.DataFrame(recovery_rows), on="event_id", validate="one_to_one")
coverage = recoveries.groupby("pace_status").agg(
    possessions=("possession_id", "size"), shot_possessions=("leads_to_shot", "sum"),
    shot_rate=("leads_to_shot", "mean"),
).reset_index()
display(coverage)


,pace_status,possessions,shot_possessions,shot_rate
0,eligible,899,359,0.399333
1,no_movement,474,278,0.586498
2,zero_duration,165,2,0.012121


## 11.4. Define pooled post-recovery pace tertiles

Define slower, middle, and faster bands from recovery possessions with measurable pace. Share boundaries across teams and preserve ties. Unknown pace stays outside the bands. Pooled results are not adjusted for source, team strength, recovery location, sequence duration, or pressure.


In [4]:
analysis_recoveries = recoveries.loc[recoveries["pace_status"].eq("eligible")].copy()
bins, pace_edges = pd.qcut(analysis_recoveries["post_recovery_pace_ft_s"], q=3,
                          labels=False, retbins=True, duplicates="drop")
band_names = {1: "Slower", 2: "Middle", 3: "Faster"}
analysis_recoveries["pace_tertile"] = (bins + 1).astype("Int64")
analysis_recoveries["pace_band"] = analysis_recoveries["pace_tertile"].map(band_names)
pace_boundaries = pd.DataFrame({
    "pace_tertile": np.arange(1, len(pace_edges)),
    "lower_pace_ft_s": pace_edges[:-1], "upper_pace_ft_s": pace_edges[1:],
})
pace_boundaries["pace_band"] = pace_boundaries["pace_tertile"].map(band_names)
display(pace_boundaries.round(2))


,pace_tertile,lower_pace_ft_s,upper_pace_ft_s,pace_band
0,1,0.00,13.04,Slower
1,2,13.04,21.41,Middle
2,3,21.41,181.91,Faster


## 11.5. Compare shot generation after recovery

Show the percentage of qualifying possessions that lead to a shot attempt. Use bars without uncertainty intervals and retain counts, games, median pace, and sequence spans. Mean first-shot distance and shot timing are conditional on generating a shot and should not be interpreted as outcomes for unsuccessful possessions.


In [5]:
recovery_summary = analysis_recoveries.groupby("pace_band").agg(
    possessions=("possession_id", "size"), shot_possessions=("leads_to_shot", "sum"),
    games=("game_id", "nunique"), shot_rate=("leads_to_shot", "mean"),
    median_pace_ft_s=("post_recovery_pace_ft_s", "median"),
    median_sequence_span_seconds=("sequence_span_seconds", "median"),
    median_time_to_shot_seconds=("time_to_first_shot_seconds", "median"),
    mean_first_shot_distance_ft=("first_shot_distance_ft", "mean"),
).reset_index()
recovery_summary["shot_pct"] = 100 * recovery_summary["shot_rate"]
display(recovery_summary.round(2))
recovery_fig = px.bar(
    recovery_summary, x="pace_band", y="shot_pct", color="pace_band",
    category_orders={"pace_band": ["Slower", "Middle", "Faster"]},
    color_discrete_map={"Slower":"#93c5fd", "Middle":"#3b82f6", "Faster":"#1d4ed8"},
    hover_data=["possessions", "shot_possessions", "games", "median_pace_ft_s", "median_sequence_span_seconds"],
    labels={"pace_band":"Post-recovery sequence pace", "shot_pct":"Possessions generating a shot (%)"},
    title="5v5 offensive-zone recovery possessions | retrospective shot generation",
)
recovery_fig.update_layout(template="plotly_white", height=450, showlegend=False)
recovery_fig.update_yaxes(range=[0,100])
recovery_fig.show()


,pace_band,possessions,shot_possessions,games,shot_rate,median_pace_ft_s,median_sequence_span_seconds,median_time_to_shot_seconds,mean_first_shot_distance_ft,shot_pct
0,Faster,300,185,34,0.62,28.83,4.0,3.0,41.08,61.67
1,Middle,297,118,34,0.40,17.18,6.0,6.0,38.79,39.73
2,Slower,302,56,34,0.19,7.91,3.0,7.0,35.54,18.54


## 11.6. Review teams and individual possessions

Retain team counts and observed sequence spans to investigate differences. Review quick failed possessions and immediate shots excluded for unknown pace as well as band-assigned possessions. Faster or slower bands can reflect different sequence lengths and measurement cutoffs.

A coaching review should inspect recovery location, available support, first actions, and shot selection. Do not infer that a team should speed up from this retrospective association alone. Consistency across games and a fixed-window sensitivity analysis would strengthen the conclusion.


In [6]:
team_recovery_summary = analysis_recoveries.groupby(["team_id", "team_name", "pace_band"]).agg(
    possessions=("possession_id", "size"), games=("game_id", "nunique"),
    shot_rate=("leads_to_shot", "mean"), median_sequence_span_seconds=("sequence_span_seconds", "median"),
).reset_index()
team_recovery_summary["shot_pct"] = 100 * team_recovery_summary["shot_rate"]
display(team_recovery_summary.round(2))

REVIEW_PACE_BAND = "Faster"
REVIEW_LEADS_TO_SHOT = False
review_recoveries = analysis_recoveries.loc[
    analysis_recoveries["pace_band"].eq(REVIEW_PACE_BAND)
    & analysis_recoveries["leads_to_shot"].eq(REVIEW_LEADS_TO_SHOT)
]
display(review_recoveries[[
    "game_id", "game_date", "team_name", "player_name", "period", "clock_seconds",
    "event_id", "possession_id", "x", "y", "post_recovery_pace_ft_s", "movement_seconds",
    "movement_transitions", "sequence_span_seconds", "passes_before_cutoff",
    "leads_to_shot", "first_shot_event_id", "time_to_first_shot_seconds",
]].head(20))


,team_id,team_name,pace_band,possessions,games,shot_rate,median_sequence_span_seconds,shot_pct
0,1,Boston Pride,Faster,26,7,0.46,4.25,46.15
1,1,Boston Pride,Middle,26,7,0.31,8.25,30.77
2,1,Boston Pride,Slower,26,7,0.23,2.75,23.08
3,2,Buffalo Beauts,Faster,19,5,0.53,3.00,52.63
4,2,Buffalo Beauts,Middle,31,6,0.35,6.00,35.48
5,2,Buffalo Beauts,Slower,29,6,0.21,3.00,20.69
6,3,Connecticut Whale,Faster,20,4,0.60,3.75,60.00
7,3,Connecticut Whale,Middle,15,4,0.20,4.50,20.00
8,3,Connecticut Whale,Slower,18,4,0.00,2.50,0.00
9,4,Metropolitan Riveters,Faster,13,3,0.46,3.00,46.15


,game_id,game_date,team_name,player_name,period,clock_seconds,event_id,possession_id,x,y,post_recovery_pace_ft_s,movement_seconds,movement_transitions,sequence_span_seconds,passes_before_cutoff,leads_to_shot,first_shot_event_id,time_to_first_shot_seconds
8,3,2021-01-23,Boston Pride,Mallory Souliotis,1,913.0,201,38,129.0,3.0,23.211587,3.000000,3,3.000000,2,False,NaN,NaN
9,3,2021-01-23,Minnesota Whitecaps,Haley Mack,1,836.0,255,47,181.0,4.0,35.608988,0.500000,2,0.500000,1,False,NaN,NaN
31,3,2021-01-23,Boston Pride,Meaghan Rickard,3,726.0,1796,333,196.0,65.0,25.514914,5.500000,4,5.500000,2,False,NaN,NaN
37,1,2021-01-23,Connecticut Whale,Maggie LaGue,1,1139.0,2353,438,137.0,82.0,78.473315,1.500000,4,1.500000,2,False,NaN,NaN
48,1,2021-01-23,Buffalo Beauts,Cassidy MacPherson,1,447.0,2836,538,138.0,20.0,24.786899,4.500000,5,4.500000,1,False,NaN,NaN
66,1,2021-01-23,Buffalo Beauts,Hunter Accursi,2,734.0,3478,672,183.0,4.0,22.184450,14.000000,14,14.000000,7,False,NaN,NaN
73,1,2021-01-23,Connecticut Whale,Shannon Doyle,2,517.0,3632,708,176.0,82.0,42.190046,0.500000,2,0.500000,1,False,NaN,NaN
79,1,2021-01-23,Buffalo Beauts,Jordan Juron,2,241.0,3824,753,142.0,4.0,21.641709,4.000000,5,4.000000,1,False,NaN,NaN
81,1,2021-01-23,Connecticut Whale,Hanna Beattie,3,1153.0,4025,799,140.0,1.0,22.634773,3.500000,4,3.500000,2,False,NaN,NaN
87,1,2021-01-23,Connecticut Whale,Shannon Doyle,3,539.0,4465,871,167.0,85.0,22.803509,0.500000,2,0.500000,1,False,NaN,NaN
